# Story 1.2: Simulation Graph E2E Test

This notebook tests the LangGraph simulation workflow end-to-end with real AI agents.

**What this tests:**
1. Complete scenario generation
2. LangGraph workflow with interrupt() for human-in-the-loop
3. Full simulation cycle through multiple turns
4. MLflow tracing of multi-step execution

## 1. Setup & Imports

In [1]:
import warnings

# Suppress MLflow/LangGraph autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig, generate_scenario_id
from summit_sim.settings import settings
from summit_sim.tracing import (
    enable_tracing,
    log_simulation_results,
    simulation_session,
)

# Initialize MLflow with unified tracing for Pydantic AI and LangGraph
enable_tracing()

print(f"✓ MLflow tracking URI: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment name: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key configured: {bool(settings.openrouter_api_key)}")
print("\n📊 MLflow Tracing Architecture:")
print("  • Scenario generation → Independent Pydantic AI traces")
print("  • Simulation loop → Nested under parent run with session_id")
print("  • class_id → Links generation and simulation traces in MLflow")
print("  • Interrupt 'exceptions' → Expected (human-in-the-loop mechanism)")

✓ MLflow tracking URI: https://mlflow.bhamm-lab.com
✓ Experiment name: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key configured: True

📊 MLflow Tracing Architecture:
  • Scenario generation → Independent Pydantic AI traces
  • Simulation loop → Nested under parent run with session_id
  • class_id → Links generation and simulation traces in MLflow
  • Interrupt 'exceptions' → Expected (human-in-the-loop mechanism)


## 2. Generate Scenario

First, we'll generate a complete scenario with multiple turns.

**Note**: Scenario generation creates an independent Pydantic AI trace in MLflow, separate from the simulation session. This is by design - generation and simulation are distinct phases.

In [2]:
# Test configuration - class_id auto-generated if not provided
config = HostConfig(num_participants=3, activity_type="hiking", difficulty="med")

print(f"Generating scenario: {config}")
print(f"📋 class_id: {config.class_id}")
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)

# Generate unique scenario ID
scenario_id = generate_scenario_id()

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Starting turn: {scenario.starting_turn_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

print("\n💡 In MLflow UI, filter by class_id to see all related traces")

Generating scenario: num_participants=3 activity_type='hiking' difficulty='med' class_id=None
📋 class_id: None
------------------------------------------------------------
✓ Title: The Ridge Warning
✓ Setting: Alpine ridgeline trail at 11,500ft (3,500m). Weather is turning cold and windy, late afternoon. Two partners are present with the patient.
✓ Patient: 32-year-old male hiker, generally fit. Complaining of severe "throbbing" headache, nausea, and recent, noticeable stumbling while walking.
✓ Turns: 4
✓ Starting turn: 0

Learning Objectives:
  • Recognize the signs and symptoms of high-altitude illness escalation (from AMS to HACE).
  • Prioritize immediate descent over non-definitive, on-site treatments.
  • Practice assisted evacuation techniques (human crutch) for a patient with ataxia.

💡 In MLflow UI, filter by class_id to see all related traces


Trace(trace_id=tr-1b243637c092d1ed0c3661941f4b25e2)

## 3. Display Scenario Turns

Let's see what turns were generated:

In [3]:
for _i, turn in enumerate(scenario.turns):
    print()
    print("=" * 60)
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"Turn {turn.turn_id} {marker}")
    print("=" * 60)
    print()
    print(turn.narrative_text)
    print()
    print("Choices:")
    for choice in turn.choices:
        if choice.next_turn_id is None:
            end = "END"
        else:
            end = f"Turn {choice.next_turn_id}"
        mark = "✓" if choice.is_correct else " "
        print(f"  [{mark}] {choice.description} -> {end}")


Turn 0 (START)

You are 3 hikers on a high-alpine pass. Suddenly, 'Alex' (32M) stops walking. He is pale, complaining of a severe, throbbing headache and feeling nauseous. When asked if he can continue, he tries to step forward but stumbles noticeably and nearly falls, despite the trail being relatively flat. He looks confused. Your current altitude is 11,500ft. The sky is darkening.

Choices:
  [✓] Immediately stop the hike and begin a rapid, assisted descent to a lower altitude. -> Turn 1
  [ ] Have the patient sit and rest for 30 minutes, administer hydration, and re-evaluate before deciding to descend. -> Turn 2

Turn 1 

You have immediately initiated descent. The patient is able to walk with the 'human crutch' technique (supported by one partner on each side). His ataxia (stumbling) is still present, and he is complaining that his headache is getting worse. The temperature is dropping, and you are still high up and exposed.

Choices:
  [✓] Continue the assisted descent as quickl

## 4. Initialize Simulation Graph

Create the LangGraph workflow with checkpointer for interrupt support:

In [4]:
# Create the graph with in-memory checkpointer
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state (TypedDict - use dict syntax)
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
}

print("✓ Graph created successfully")
print(f"✓ Initial state: turn_id={initial_state['current_turn_id']}")
print(f"✓ class_id: {initial_state['class_id']}")
print("\n⚠️  Next cell will start MLflow session with parent run tracking")

✓ Graph created successfully
✓ Initial state: turn_id=0
✓ class_id: None

⚠️  Next cell will start MLflow session with parent run tracking


## 5. Run Simulation with MLflow Session Tracking

Execute the full simulation workflow within an MLflow session. The simulation will:
1. Create a parent MLflow run with session metadata
2. Present each turn (with traces nested under parent)
3. Pause with `interrupt()` for your choice
4. Call the AI agent for feedback
5. Advance to next turn
6. Log final results to the parent run

**MLflow Features:**
- Session ID links all traces together
- Parent run contains all agent calls as child spans
- Automatic error tracking (status: completed/failed)
- Final metrics: total_turns, is_complete, learning_moments_count

**⚠️ Expected Behavior:**
- `interrupt()` raises exceptions for human-in-the-loop - these appear in MLflow traces but are NOT errors
- Scenario generation (Cell 2) creates separate Pydantic AI traces
- Only simulation traces are nested under the parent run

In [5]:
from langgraph.types import Command

# Wrap simulation in session context for unified MLflow tracking
with simulation_session(config, scenario_id=scenario_id) as (session_id, graph_config):
    print(f"\n📋 Session ID: {session_id}")
    print(
        f"📊 Session Name: sim-{config.activity_type}-{config.num_participants}p-{config.difficulty}-...\n"
    )

    current_state = initial_state
    turn_count = 0
    simulation_complete = False

    print("\n=== STARTING SIMULATION ===\n")

    while not simulation_complete:
        turn_count += 1
        print(f"--- TURN {turn_count} ---")

        async for event in graph.astream(current_state, graph_config):
            if "__interrupt__" in event:
                interrupt_obj = event["__interrupt__"]
                if hasattr(interrupt_obj, "value"):
                    interrupt_data = interrupt_obj.value
                elif isinstance(interrupt_obj, (list, tuple)):
                    interrupt_data = (
                        interrupt_obj[0].value
                        if hasattr(interrupt_obj[0], "value")
                        else interrupt_obj[0]
                    )
                else:
                    interrupt_data = interrupt_obj

                print(f"\n📖 {interrupt_data['narrative']}\n")
                print("Choices:")
                for i, choice in enumerate(interrupt_data["choices"], 1):
                    print(f"  {i}. {choice['description']}")

                while True:
                    try:
                        sel = int(input("\nSelect: ")) - 1
                        if 0 <= sel < len(interrupt_data["choices"]):
                            break
                        print("Invalid")
                    except ValueError:
                        print("Enter number")

                selected = interrupt_data["choices"][sel]
                current_state = await graph.ainvoke(
                    Command(resume={"choice_id": selected["choice_id"]}), graph_config
                )
                break

            # Check for graph completion
            if event == {}:
                simulation_complete = True
                break

        # Check if state indicates completion (AppState is TypedDict, use dict access)
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("Safety stop")
            break

    print("\n=== SIMULATION COMPLETE ===")

    # Log final results to MLflow parent run
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
    )
    print("\n✓ Results logged to MLflow")


📋 Session ID: e8a3f709-ae67-4a31-bc04-8e2584e063fb
📊 Session Name: sim-hiking-3p-med-...


=== STARTING SIMULATION ===

--- TURN 1 ---


2026/03/23 20:00:36 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fbb26bd3600> at 0x7fbae61bd7c0> was created in a different Context
2026/03/23 20:00:36 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fbb26bd3600> at 0x7fbae694b4c0> was created in a different Context



📖 You are 3 hikers on a high-alpine pass. Suddenly, 'Alex' (32M) stops walking. He is pale, complaining of a severe, throbbing headache and feeling nauseous. When asked if he can continue, he tries to step forward but stumbles noticeably and nearly falls, despite the trail being relatively flat. He looks confused. Your current altitude is 11,500ft. The sky is darkening.

Choices:
  1. Immediately stop the hike and begin a rapid, assisted descent to a lower altitude.
  2. Have the patient sit and rest for 30 minutes, administer hydration, and re-evaluate before deciding to descend.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/23 20:00:49 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fbb26bd3600> at 0x7fbae6384780> was created in a different Context
2026/03/23 20:00:57 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fbb26bd3600> at 0x7fbae622a000> was created in a different Context
2026/03/23 20:00:57 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fbb26bd3600> at 0x7fbae62543c0> was created in a different Context
2026/03/23 20:00:57 WARNING mlflow.utils.autologging_utils: Encountered un

--- TURN 2 ---


...

--- TURN 4 ---

📖 You are successfully descending. The patient is still disoriented, but being lower is clearly impacting his symptoms positively. You still have several hundred feet of vertical descent to reach the safety of the tree line. The trail ahead is well-marked but rocky. Light is fading.

Choices:
  1. Continue the descent until you reach the tree line (significantly lower altitude) before calling for SAR or resting.
  2. Stop at the first available flat spot to signal for a helicopter rescue via PLB, as descent is too difficult.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/23 20:01:39 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fbb26bd3600> at 0x7fbae59a0140> was created in a different Context
2026/03/23 20:01:46 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


=== SIMULATION COMPLETE ===

✓ Results logged to MLflow
🏃 View run sim-noclass-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/52715d6edd5142db8a7d9f57db9938ef
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-ac8d52039ebc36cd916b11772708e60f), Trace(trace_id=tr-6224dfefbaff58ff269729b94d4dd2f5), Trace(trace_id=tr-8618e6074e2cc12ae3de07a8cfe472b4), Trace(trace_id=tr-f38b3c740b85ca8bbe743bbc5e95a475)]

2026/03/23 20:01:46 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7fbb26bd3600> at 0x7fbae6372f80> was created in a different Context


## 6. Display Results

Show the complete transcript and learning moments:

In [6]:
print("\n" + "=" * 60)
print("TRANSCRIPT")
print("=" * 60 + "\n")

for _i, entry in enumerate(current_state["transcript"], 1):
    print(f"Turn {entry['turn_id']}:")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print(f"  Learning: {', '.join(entry['learning_moments'])}")
    print()


TRANSCRIPT

Turn 0:
  Situation: You are 3 hikers on a high-alpine pass. Suddenly, 'Alex' (32M) stops walking. He...
  Choice: Immediately stop the hike and begin a rapid, assisted descent to a lower altitude.
  Feedback: Excellent decision. By recognizing the patient's ataxia as a sign of critical neurological impairmen...
  Learning: Ataxia (uncoordinated gait) at high altitude is a classic indicator of High Altitude Cerebral Edema (HACE) and necessitates immediate, active descent., In cases of suspected HACE, do not rely on resting or hydration to see if symptoms resolve, as this wastes essential time and does not provide definitive treatment.

Turn 0:
  Situation: You are 3 hikers on a high-alpine pass. Suddenly, 'Alex' (32M) stops walking. He...
  Choice: Immediately stop the hike and begin a rapid, assisted descent to a lower altitude.
  Feedback: Excellent decision. By recognizing the patient's ataxia as a sign of critical neurological impairmen...
  Learning: Ataxia (uncoordin

In [7]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for _i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{_i}. {moment}")


KEY LEARNING MOMENTS

1. Ataxia (uncoordinated gait) at high altitude is a classic indicator of High Altitude Cerebral Edema (HACE) and necessitates immediate, active descent.
2. In cases of suspected HACE, do not rely on resting or hydration to see if symptoms resolve, as this wastes essential time and does not provide definitive treatment.
3. Ataxia (uncoordinated gait) at high altitude is a classic indicator of High Altitude Cerebral Edema (HACE) and necessitates immediate, active descent.
4. In cases of suspected HACE, do not rely on resting or hydration to see if symptoms resolve, as this wastes essential time and does not provide definitive treatment.
5. Ataxia (uncoordinated gait) at high altitude is a classic indicator of High Altitude Cerebral Edema (HACE) and necessitates immediate, active descent.
6. In cases of suspected HACE, do not rely on resting or hydration to see if symptoms resolve, as this wastes essential time and does not provide definitive treatment.
7. Ataxia (

## 7. MLflow Verification

Check your MLflow UI to verify:

### Trace Architecture
- **Scenario Generation** (Cell 2): Independent Pydantic AI trace
  - Tagged with `class_id` for linking to simulation
  - Appears as standalone trace in MLflow

- **Simulation** (Cell 5): Parent run with nested LangGraph traces
  - Parent Run: `sim-{class_id}-{activity}-{participants}p-{difficulty}`
  - Tagged with same `class_id` as generation
  - Session ID: UUID used for thread_id grouping
  - Child spans: Each agent call nested under parent

### Linking Generation and Simulation
Both phases share the same `class_id` tag:
- Filter in MLflow UI: `tags.class_id = "{class_id}"`
- Or search: `{class_id}` in run names
- This links: generation trace → simulation parent run

### Expected Behaviors
- **Interrupt 'Exceptions'**: Normal! LangGraph's `interrupt()` raises exceptions
  for human-in-the-loop. These appear in traces but are not errors.
- **Evaluation Tab**: Runs with traces appear here in MLflow 3.x UI
- **Session View**: Filter by thread_id to see grouped traces

### Logged Data
- **Metrics**: total_turns, is_complete, learning_moments_count
- **Tags**: class_id, session_type, activity_type, difficulty, status
- **Params**: class_id, session_id (UUID), activity_type, num_participants, difficulty

In [8]:
import mlflow

# Show experiment info
experiment = mlflow.get_experiment_by_name(settings.mlflow_experiment_name)
print(f"Experiment ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"\nView traces at: {settings.mlflow_tracking_uri}")
print("\nIn MLflow UI, look for:")
print(f"  📁 class_id: {config.class_id}")
print(f"  📊 Generation trace: Filter by class_id tag or search '{config.class_id}'")
print(f"  📊 Simulation run: sim-{config.class_id}-... (parent run with nested traces)")
print("\nKey Features:")
print(f"  • class_id '{config.class_id}' links generation and simulation")
print('  • Filter: tags.class_id = "<your-class-id>"')
print("  • Metrics logged at session completion")
print("  • Status tag shows completed/failed")

Experiment ID: 7
Artifact Location: mlflow-artifacts:/7

View traces at: https://mlflow.bhamm-lab.com

In MLflow UI, look for:
  📁 class_id: None
  📊 Generation trace: Filter by class_id tag or search 'None'
  📊 Simulation run: sim-None-... (parent run with nested traces)

Key Features:
  • class_id 'None' links generation and simulation
  • Filter: tags.class_id = "<your-class-id>"
  • Metrics logged at session completion
  • Status tag shows completed/failed


---
## ✅ E2E Test Complete

If you successfully completed the simulation:
- ✓ LangGraph workflow executed correctly
- ✓ Human-in-the-loop interrupt worked
- ✓ AI feedback generated for each choice
- ✓ State advanced through all turns
- ✓ MLflow captured multi-step execution